In [1]:
# %% [0] Setup — metric, Weibull, postprocess
import re, io, time, unicodedata
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from scipy.optimize import minimize
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(0); np.random.seed(0)

INPUT_DIR = sorted(Path("/kaggle/input").glob("**/Training_labels_updated.csv"))[0].parent
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

SIZES = np.array([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2, 6.3, 20, 63, 200])
SIZE_COLS = ["0.002","0.0063","0.02","0.063","0.2","0.63","2","6.3","20","63","200"]
W_EMD = np.diff(np.log10(SIZES))          # exact competition weights, left-endpoint rule
CAM_EDGE, CAM_A52 = "Motorola Edge", "Samsung A52"
TARGET_PPM, TRAIN_LIKE_PPM, CROP_PX = 4.6, 4.6, 400

def emd(y_true, y_pred):
    return (np.abs(np.asarray(y_true)[..., :10] - np.asarray(y_pred)[..., :10]) * W_EMD).sum(-1)

def postprocess(curve):
    c = np.maximum.accumulate(np.clip(np.asarray(curve, float), 0, 100), axis=-1)
    c[..., -1] = 100.0
    return c

def weibull_curve(lnb, c, d=SIZES):
    lnb = np.asarray(lnb, float)[..., None]; c = np.maximum(np.asarray(c, float), 0.02)[..., None]
    return 100.0 * (1.0 - np.exp(-np.exp(c * (np.log(d) - lnb))))

# %% [1] Index photos — robust recursive glob, physical scale from the file's OWN size
CAMERA_RE = re.compile(r"^(Motorola_Edge_60_fusion|Motorola_Edge|Samsung_A52|iPhone_?14|iPhone_?16)_(.+?)(?:_\d+| \(\d+\))?$")
CAMERA_KEY = {"Motorola_Edge_60_fusion": "Motorola Edge 60 Fusion", "Motorola_Edge": CAM_EDGE,
              "Samsung_A52": CAM_A52, "iPhone14": "iPhone 14", "iPhone_14": "iPhone 14",
              "iPhone16": "iPhone 16", "iPhone_16": "iPhone 16"}

def normalize_id(raw):
    s = unicodedata.normalize("NFC", raw).replace("ü","ue").replace("ö","oe").replace("ä","ae")
    return s.replace(",", "_")

labels = pd.read_csv(INPUT_DIR/"Training_labels_updated.csv", dtype={"sample_id": str}).set_index("sample_id")
sample_sub = pd.read_csv(INPUT_DIR/"sample_submission.csv", dtype={"sample_id": str})
ppm_tab = pd.read_csv(INPUT_DIR/"ppm_updated.csv").set_index("phone")

rows = []
for split, pattern in [("train", "Training-All_Photos*"), ("test", "Test_All_Photos*")]:
    for p in sorted(INPUT_DIR.glob(f"{pattern}/**/*")):
        if p.suffix.lower() not in {".jpg", ".jpeg"}: continue
        m = CAMERA_RE.match(unicodedata.normalize("NFC", p.stem))
        if m is None: raise ValueError(f"unexpected file name: {p.name}")
        with Image.open(p) as im: w, h = im.size
        rows.append({"split": split, "path": p, "camera": CAMERA_KEY[m.group(1)],
                     "sample_id": normalize_id(m.group(2)), "w": w, "h": h})
photos = pd.DataFrame(rows)
# ppm from the file's OWN current size, proportional to the table's reference size —
# correct whether or not this file was downsized before upload (no native-resolution lookup needed)
photos["ppm_eff"] = [ppm_tab.loc[c,"ppm"] * max(w,h) / ppm_tab.loc[c,"width"]
                     for c,w,h in zip(photos.camera, photos.w, photos.h)]
assert sorted(set(photos.query("split=='train'").sample_id)) == sorted(labels.index)
assert set(photos.query("split=='test'").sample_id) == set(sample_sub.sample_id)
tr, te = photos[photos.split=="train"], photos[photos.split=="test"]
print(photos.groupby(["split","camera"]).agg(photos=("path","size"), px_per_mm=("ppm_eff","median")).round(1))

# %% [2] Fit (ln b, c) training targets by minimizing the EXACT metric (for Branch A only)
_LOG_SIZES = np.log(SIZES[:10])
def fit_weibull_label(y10):
    g_lnb, g_c = np.linspace(-6,4,41), np.linspace(0.1,2.0,39)
    L, C = np.meshgrid(g_lnb, g_c, indexing="ij")
    F = 100*(1-np.exp(-np.exp(C[...,None]*(_LOG_SIZES-L[...,None]))))
    loss = (np.abs(F - y10) * W_EMD).sum(-1)
    i, j = np.unravel_index(loss.argmin(), loss.shape)
    res = minimize(lambda p: (np.abs(100*(1-np.exp(-np.exp(p[1]*(_LOG_SIZES-p[0]))))-y10)*W_EMD).sum(),
                    [L[i,j], C[i,j]], method="Nelder-Mead")
    return res.x

wb_labels = labels[SIZE_COLS[:10]].apply(lambda r: pd.Series(fit_weibull_label(r.values), index=["lnb","c"]), axis=1)
print(wb_labels.describe().round(2))

# %% [3] Shared preprocessing — physical scale + degrade sharp test photos toward train condition
def load_gray(path, ppm_eff, size_px=None):
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    a = np.asarray(im)
    if ppm_eff > 6.0:                               # sharp (test) photo -> degrade toward train condition
        f = TRAIN_LIKE_PPM / ppm_eff
        a = cv2.resize(a, (round(a.shape[1]*f), round(a.shape[0]*f)), interpolation=cv2.INTER_AREA)
        buf = io.BytesIO(); Image.fromarray(a).save(buf, format="JPEG", quality=75)
        a = np.asarray(Image.open(io.BytesIO(buf.getvalue())))
        ppm_eff = TRAIN_LIKE_PPM
    f = TARGET_PPM / ppm_eff
    a = cv2.resize(a, (round(a.shape[1]*f), round(a.shape[0]*f)), interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]: a = np.rot90(a)
    H, W = a.shape[:2]
    if size_px:                                      # fixed-size centre crop for the CNN
        s = min(size_px, H, W); y, x = (H-s)//2, (W-s)//2
        a = cv2.resize(a[y:y+s, x:x+s], (size_px, size_px), interpolation=cv2.INTER_AREA)
        return a
    c = int(0.25*min(H,W))                            # 50% centre crop for DoG
    return cv2.cvtColor(a[c:H-c, c:W-c], cv2.COLOR_RGB2GRAY)

DOG_SIGMAS_MM = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
def dog_features(gray):
    blurs = [gray.astype(np.float32)] + [cv2.GaussianBlur(gray.astype(np.float32), (0,0), s*TARGET_PPM) for s in DOG_SIGMAS_MM]
    return np.array([np.abs(blurs[i]-blurs[i+1]).mean() for i in range(len(DOG_SIGMAS_MM))])

# %% [4] Branch A — DoG texture -> Ridge -> Weibull (fast, robust, linear-on-purpose)
def dog_table(df):
    feats = [dog_features(load_gray(r.path, r.ppm_eff)) for r in df.itertuples()]
    return pd.DataFrame(feats, index=df.sample_id, columns=[f"dog_{i}" for i in range(len(DOG_SIGMAS_MM))])

Xa_tr_photo = dog_table(tr)
scaler_a = StandardScaler().fit(Xa_tr_photo.values)
ridge = Ridge(alpha=1.0).fit(scaler_a.transform(Xa_tr_photo.values), wb_labels.loc[Xa_tr_photo.index].values)

def branch_a_predict(df):
    Xp = dog_table(df)
    pred = ridge.predict(scaler_a.transform(Xp.values))
    curves = pd.DataFrame(weibull_curve(pred[:,0], pred[:,1]), index=Xp.index, columns=SIZE_COLS)
    return postprocess(curves.groupby(level=0).mean().to_numpy()), curves.index.unique()

predA_tr_curve, _ = branch_a_predict(tr)
print("Branch A (DoG+Ridge) self-fit EMD on train:", emd(labels.loc[sorted(set(tr.sample_id)), SIZE_COLS].values, predA_tr_curve).mean().round(2))

# %% [5] Branch B — EfficientNetV2S (frozen) + trainable head, trained directly on the real metric
eff = tf.keras.applications.EfficientNetV2S(include_top=False, weights="imagenet", pooling="avg")
eff.trainable = False
inp = tf.keras.Input((CROP_PX, CROP_PX, 3))
x = tf.keras.applications.efficientnet_v2.preprocess_input(inp)
x = eff(x, training=False)
for n in (2048, 512, 256, 128):
    x = tf.keras.layers.Dense(n, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.1)(x)
out = tf.keras.layers.Dense(2, activation="linear")(x)       # raw (ln_b, c_raw)
model = tf.keras.Model(inp, out)

D_SUPPORT = tf.constant(SIZES[:10], dtype=tf.float32)
W_TF = tf.constant(W_EMD, dtype=tf.float32)
def emd_loss(y_true, y_pred):                                  # trains directly on the EXACT metric, no
    ln_b, c = y_pred[:,0], tf.nn.softplus(y_pred[:,1]) + 0.02   # intermediate Weibull-label-fitting step
    ratio = D_SUPPORT[None,:] / tf.exp(ln_b)[:,None]
    F = 100.0 * (1.0 - tf.exp(-tf.pow(ratio, c[:,None])))
    return tf.reduce_sum(tf.abs(F - y_true) * W_TF[None,:], axis=1)

def aug_load(path, ppm_eff, y):
    a = load_gray(path, ppm_eff, size_px=CROP_PX)
    return a.astype(np.float32), y

def make_dataset(df, training):
    paths = df.path.values; ppms = df.ppm_eff.values
    has_labels = all(sid in labels.index for sid in df.sample_id)
    ys = (labels.loc[df.sample_id, SIZE_COLS[:10]].values.astype(np.float32)
          if has_labels else np.zeros((len(df), 10), dtype=np.float32))  # placeholder for test (never trained on)

    def gen():
        for p, ppm, y in zip(paths, ppms, ys):
            a, yy = aug_load(p, ppm, y)
            yield a, yy

    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((CROP_PX,CROP_PX,3), tf.float32), tf.TensorSpec((10,), tf.float32)))

    if training:
        def augment(a, y):
            a = tf.image.random_flip_left_right(a); a = tf.image.random_flip_up_down(a)
            a = tf.image.rot90(a, tf.random.uniform([], 0, 4, dtype=tf.int32))
            a = tf.image.random_brightness(a, 0.15); a = tf.image.random_contrast(a, 0.85, 1.15)
            return a, y
        ds = ds.repeat().map(augment, num_parallel_calls=tf.data.AUTOTUNE)   # infinite, not repeat(15)
    return ds.batch(16).prefetch(tf.data.AUTOTUNE)
    
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=emd_loss)
steps = max(1, len(tr) // 16)
model.fit(make_dataset(tr, training=True), epochs=25, steps_per_epoch=steps,
          callbacks=[tf.keras.callbacks.EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)])
def branch_b_predict(df):
    ds = make_dataset(df, training=False)
    pred = model.predict(ds, verbose=0)
    ln_b, c = pred[:,0], np.log1p(np.exp(pred[:,1])) + 0.02
    curves = pd.DataFrame(weibull_curve(ln_b, c), index=df.sample_id, columns=SIZE_COLS)
    return postprocess(curves.groupby(level=0).mean().to_numpy()), curves.index.unique()

predB_tr_curve, _ = branch_b_predict(tr)
print("Branch B (EfficientNet, train-fit) EMD on train:", emd(labels.loc[sorted(set(tr.sample_id)), SIZE_COLS].values, predB_tr_curve).mean().round(2))

# %% [6] Blend A + B in CURVE space (safe: always valid after postprocess either way)
# Start at 0.5/0.5; re-weight using a cross-phone check (train on one camera, validate on the
# other) before trusting this on the real leaderboard — the train-only numbers above are optimistic.
ALPHA_B = 0.5
sids_tr = sorted(set(tr.sample_id))
blend_tr = postprocess(ALPHA_B*predB_tr_curve + (1-ALPHA_B)*predA_tr_curve)
print("Blend EMD on train:", emd(labels.loc[sids_tr, SIZE_COLS].values, blend_tr).mean().round(2))

predA_te, idsA = branch_a_predict(te)
predB_te, idsB = branch_b_predict(te)
assert list(idsA) == list(idsB)
pred_A_only = pd.DataFrame(predA_te, index=idsA, columns=SIZE_COLS)
pred_blend = pd.DataFrame(postprocess(ALPHA_B*predB_te + (1-ALPHA_B)*predA_te), index=idsA, columns=SIZE_COLS)

# %% [7] SAM — measure cobbles directly, redraw the 20/63 mm points where stones are visible
import torch
from transformers import pipeline
mask_gen = pipeline("mask-generation", model="facebook/sam-vit-base", device=0 if torch.cuda.is_available() else -1)
WORK_PPM, WORK_CROP = 2.0, 0.12

def load_work(path, ppm):
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    a = cv2.resize(np.asarray(im), None, fx=WORK_PPM/ppm, fy=WORK_PPM/ppm, interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]: a = np.rot90(a)
    H, W = a.shape[:2]; c = WORK_CROP
    return np.ascontiguousarray(a[int(H*c):int(H*(1-c)), int(W*c):int(W*(1-c))])

def stones_from_masks(masks, tot):
    order = sorted(range(len(masks)), key=lambda i: -int(masks[i].sum())); taken = np.zeros_like(masks[0], bool)
    out = []
    for i in order:
        m = masks[i]; sz = int(m.sum())
        if sz < (8.0*WORK_PPM)**2 or sz > 0.5*tot or (m & taken).sum() > 0.5*sz: continue
        ys, xs = np.nonzero(m); major = max(ys.max()-ys.min(), xs.max()-xs.min()) / WORK_PPM
        fill = sz / (np.pi*(major*WORK_PPM/2)**2 + 1e-6)
        if fill < 0.35: continue
        taken |= m; out.append({"area_frac": sz/tot, "major_mm": major})
    return out

targets = photos[photos.ppm_eff >= 12.0]              # full-res only: test photos + H374 calibration sample
meas = []
for r in targets.itertuples():
    rgb = load_work(r.path, r.ppm_eff)
    out = mask_gen(Image.fromarray(rgb), points_per_batch=64, pred_iou_thresh=0.86, stability_score_thresh=0.9)
    stones = stones_from_masks([np.asarray(m) for m in out["masks"]], rgb.shape[0]*rgb.shape[1])
    meas.append({"sample_id": r.sample_id,
                "A63": sum(s["area_frac"] for s in stones if s["major_mm"] > 63),
                "A20": sum(s["area_frac"] for s in stones if s["major_mm"] > 20)})
ms = pd.DataFrame(meas).groupby("sample_id").mean()

I20, I63 = SIZE_COLS.index("20"), SIZE_COLS.index("63")
if "H374" in ms.index:
    K_CAL = (100.0 - labels.loc["H374","20"]) / (100.0 * ms.loc["H374","A20"])
else:
    K_CAL = 1.0  # Delesse fallback: area fraction ~ mass fraction, if the calibration sample wasn't measured
print(f"k = {K_CAL:.3f}")

def redraw(curve, a63, a20):
    c = curve.copy()
    new20 = max(100.0 - K_CAL*100.0*a20, 0.0); new63 = max(100.0 - K_CAL*100.0*a63, new20)
    scale = new20 / max(c[I20], 1e-6)
    if scale < 1: c[:I20+1] *= scale
    c[I20], c[I63], c[-1] = new20, new63, 100.0
    return c

final = {}
for sid in pred_blend.index:
    a63, a20 = (float(ms.loc[sid,"A63"]), float(ms.loc[sid,"A20"])) if sid in ms.index else (0.0, 0.0)
    final[sid] = redraw(pred_blend.loc[sid].to_numpy(), a63, a20) if a63 >= 0.02 else pred_blend.loc[sid].to_numpy()
pred_final = pd.DataFrame(postprocess(np.stack([final[s] for s in pred_blend.index])), index=pred_blend.index, columns=SIZE_COLS)

# %% [8] Submission + structural checks
sub = sample_sub[["sample_id"]].merge(pred_final, left_on="sample_id", right_index=True, how="left")
vals = sub[SIZE_COLS].to_numpy()
assert not np.isnan(vals).any()
assert ((vals>=0)&(vals<=100)).all() and (np.diff(vals,axis=1)>=0).all() and (vals[:,-1]==100).all()
sub.to_csv(OUTPUT_DIR/"submission.csv", index=False, float_format="%.4f")
pred_A_only.reset_index().rename(columns={"index":"sample_id"}).to_csv(OUTPUT_DIR/"submission_branchA_only.csv", index=False)
print(sub.round(1).to_string(index=False))

                               photos  px_per_mm
split camera                                    
test  iPhone 14                    14       13.9
      iPhone 16                    21       19.5
train Motorola Edge                69        4.6
      Motorola Edge 60 Fusion       3       12.5
      Samsung A52                  55        4.6
         lnb      c
count  24.00  24.00
mean   -0.17   0.86
std     2.27   0.46
min    -3.56   0.37
25%    -2.42   0.59
50%     0.81   0.79
75%     2.08   0.94
max     2.42   2.71
Branch A (DoG+Ridge) self-fit EMD on train: 36.37


I0000 00:00:1793289739.956479      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793289739.960976      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/25


2026-10-29 16:03:07.050958: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-29 16:03:07.200927: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-29 16:03:08.194135: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-29 16:03:08.351129: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-29 16:03:09.975351: E external/local_xla/xla/stream_

7/7 ━━━━━━━━━━━━━━━━━━━━ 69s 370ms/step - loss: 88.9591
Epoch 2/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 393ms/step - loss: 51.5326
Epoch 3/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 446ms/step - loss: 50.9827
Epoch 4/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 454ms/step - loss: 50.7974
Epoch 5/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 468ms/step - loss: 49.5686
Epoch 6/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 467ms/step - loss: 44.7873
Epoch 7/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 477ms/step - loss: 43.6978
Epoch 8/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 479ms/step - loss: 39.2693
Epoch 9/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 293ms/step - loss: 41.2253
Epoch 10/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 378ms/step - loss: 39.2301
Epoch 11/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 468ms/step - loss: 38.9651
Epoch 12/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 461ms/step - loss: 34.2232
Epoch 13/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 392ms/step - loss: 38.9356
Epoch 14/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 438ms/step - loss: 37.6610
Epoch 15/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 434ms/step - loss: 40.2742
Epoch 16/25
7/

2026-10-29 16:04:49.555854: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-29 16:04:49.704909: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-29 16:04:50.702732: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-29 16:04:50.857196: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-29 16:04:52.481827: E external/local_xla/xla/stream_

... (5 outputs left out)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


k = 1.305
               sample_id  0.002  0.0063  0.02  0.063  0.2  0.63    2   6.3    20    63   200
        HPC_Airbus BS6-3    2.1     4.9  11.6   25.8 51.6  82.6 98.5 100.0 100.0 100.0 100.0
          HPC_Audorfring    3.5     7.1  14.3   27.7 50.3  77.7 95.4  99.7 100.0 100.0 100.0
HPC_Muenster_BS6_9_0-10m    0.2     0.4   0.8    1.5  2.6   4.1  5.2   5.5   5.5  65.2 100.0
   HPC_Airbus BS10-4bis7    1.1     2.5   5.6   12.4 26.3  50.1 78.6  95.6  99.7 100.0 100.0
   HPC_Airbus BS12-5bis8    2.6     5.6  12.0   24.5 46.5  74.8 95.0  99.8 100.0 100.0 100.0
 HPC_Kleinkummerfeld 2-2    2.5     5.2  10.9   22.0 41.8  69.1 92.0  99.5 100.0 100.0 100.0
 HPC_Kleinkummerfeld 2-3   26.3    27.3  28.5   30.4 33.6  39.3 49.2  63.4  76.9  82.6 100.0
 HPC_Kleinkummerfeld 9-4    2.8     4.2   6.5   10.3 16.9  28.0 45.7  68.1  87.0  95.4 100.0
HPC_Kleinkummerfeld 18-3    2.0     5.0  12.0   27.3 54.9  85.6 98.9 100.0 100.0 100.0 100.0
   HPC_Testfeld Lidl WHV    3.2     6.4  12.9   25.1 45.9  7